In [1]:
%run pi1.ipynb
%run monodromyfunctions.ipynb
%run ptolemy_equations.ipynb\
%run Matrix.ipynb

def apply_dict_to_str(dictionary:dict, word:str):
    for key in dictionary.keys():
        word = word.replace(key, dictionary[key])
    return word




#This class will compute all boundary parabolic representations of the given PTB and tensor them with the abelianization homomorphism.
class Representations():
    def __init__(self, monodromy:str):
        self.monodromy = monodromy
        self.pi1 = Pi1(monodromy)
        self.h1 = self.pi1.homology
        
        self.ptolemy = PtolemyEquations(self.monodromy)
        self.coord_ring = self.ptolemy.list_all_coord_ring() 
        
        # self.h1_group_rings = self.make_group_rings(self.coord_ring)
        self.ptolemy_coordinates_ring = LaurentPolynomialRing(QQ, 't_', len(self.monodromy)+1)
        self.h1_group_ring = self.h1.algebra(self.ptolemy_coordinates_ring)
        
        
        temp_gluing_dict = self.ptolemy.gluing_dict
        temp_gluing_dict['t_0 '] = '1'
        self.gluing_dict = temp_gluing_dict
        
        
        
        
        
        #for the sake of cleaning up some of this code, we refer to each obstruction class in an array as:
        #0 - trivial
        #1 - monodromy
        #2 - meridean
        #3 - longitude
        #4 - mon-mer
        #5 - mon-long
        #6 - mer-long
        #7 - all
        
        
        
    def make_group_rings(self, rings):
        out_rings = []
        
        for i in rings:
            if i[1] != 0:
                try:
                    f = i[1].fraction_field()
                    out_rings.append(self.h1.algebra(f))
                    
                except:
                    out_rings.append(0)
            else:
                out_rings.append(0)

        return out_rings
    
    
    def make_matrix_space(self, ring):
        
        return MatrixSpace(ring, 2, 2)
    
    
    def repeat_gluing_dict(self, word:str):
        end = len(self.monodromy)
    
        for w in range(end):
            word = apply_dict_to_str(self.gluing_dict, word)
        
        return word
    
    #Make a dictionary which replaces string values with entries in the ring. This allows us to cast the matrix of strings into a matrix of ring elements.
    
    def string_to_ring_dict(self, ring):
        end = len(self.monodromy) + 1
        dictionary = {}
        
        
        for w in range(end):
            dictionary[f't_{w} '] = {ring.gens()[w]}
        return dictionary
    
    
    
    def replace_in_matrix(self, matrix):
        end = len(self.monodromy) - 1

        for i in range(2):
            for j in range(2):
                matrix[i][j] = self.repeat_gluing_dict(matrix[i][j])
            
        return matrix
    
    def make_generic_mer_representation(self):
        end = len(self.monodromy) - 1
        R = self.ptolemy_coordinates_ring
        
        mat = self.replace_in_matrix([[f'1 + (b_0 )', f'1'], [f'(t_{end} )', '0']])

        return matrix(self.h1_group_ring, [[R(mat[0][0])*R(self.repeat_gluing_dict('v_0 '))^-1, R(f'-t_{end}')^-1],[R(f't_{end}'), 0]]) * self.h1_group_ring.gen(0)
    
    
    def make_generic_long_representation(self):
        end = len(self.monodromy) - 1
        R = self.ptolemy_coordinates_ring
        
        mat = self.replace_in_matrix([['1', '-(t_0 )*(b_0 ) - (h_0 )^2'], ['(v_0 )', '-(b_0 )']])
        
        return matrix(self.h1_group_ring, [[R(f't_{end}')^-1, R(mat[0][1])*R(f't_{end}')^(-2)],[R(mat[1][0]), R(mat[1][1])*R(f't_{end}')^-1]]) * self.h1_group_ring.gen(1)
    
    #For the monodromy representation, when we transfer from R to L or L to R, we pick up a matrix. This will be an upper right triangular matrix with 1s on the diagonal.
    
    def hinge(self, index):
        R = self.ptolemy_coordinates_ring
        sign = 0
        if self.monodromy[index] == 'R':
            sign = 1
        else:
            sign = -1
        
        return matrix(self.h1_group_ring, [[1, sign * R(self.repeat_gluing_dict(f'b_{index} '))*R(self.repeat_gluing_dict(f'h_{index} * v_{index} '))^-1],[0, 1]])
    
    def compare_to_last(self, index):
        return self.monodromy[index] != self.monodromy[index - 1]
        
    def make_generic_monodromy_representation(self):
        end = len(self.monodromy)
        
        monodromy = identity_matrix(2)
        
        for w in range(end):
            if self.compare_to_last(w):
                monodromy = self.hinge(w) * monodromy
        
        return monodromy * self.h1_group_ring.gen(2)
        